# Análisis de los hábitos de escucha en Spotify

Cuaderno de análisis del proyecto final de Técnicas de Ciencia de Datos. Parte de los dos CSV generados con `TrabajoTecnicasDeCienciaDeDatos.ipynb` y desarrolla el modelo de datos, el enriquecimiento, la compatibilidad entre usuarios, la afinidad con artistas nuevos y el agrupamiento por géneros.

## Orden de ejecución

1. Ejecutar el cuaderno de extracción con la cuenta de cada autor para generar `top_artistas.csv` y `top_artistasJorge.csv`.
2. Ejecutar este cuaderno completo con *Restart & Run All*.

## Privacidad

Los CSV crudos, la caché de enriquecimiento y las tablas con nombres de artistas no se versionan. Solo se publican agregados (`resultados/*.csv` y `resultados/metricas.json`) y las figuras del informe.


In [ ]:
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

import spotify_analisis as sa

sns.set_theme(style="whitegrid", context="notebook")
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
COLORES_USUARIO = {"pablo": PALETA[0], "jorge": PALETA[1]}
SUPERFICIE = "#fcfcfb"
SECUENCIAL = LinearSegmentedColormap.from_list("azul", ["#f2f6fc", "#86b6ef", "#2a78d6", "#0d366b"])
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
    "axes.titlesize": 12, "axes.labelsize": 10,
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE,
    "text.color": "#0b0b0b", "axes.labelcolor": "#0b0b0b",
    "axes.edgecolor": "#c3c2b7", "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
})
DIR_RES = Path("resultados")
DIR_RES.mkdir(exist_ok=True)
RUTAS = {"pablo": "top_artistas.csv", "jorge": "top_artistasJorge.csv"}
CORTOS = {"ultimas_4_semanas": "4 semanas", "ultimos_6_meses": "6 meses", "varios_anios": "Varios años", "global": "Global"}

## 1. Carga y validación

In [ ]:
df = sa.cargar_datos(RUTAS)
validacion = sa.validar_datos(df)
validacion.to_csv(DIR_RES / "validacion.csv", index=False)
display(validacion)
print(f"Filas: {len(df)}; artistas únicos: {df['artista'].nunique()}")

## 2. Consolidación y géneros

In [ ]:
generos = sa.explosionar_generos(df)
resumen = sa.resumen_por_usuario_rango(df)
principales = sa.genero_principal(generos)
resumen.to_csv(DIR_RES / "resumen_datos.csv", index=False)
display(resumen)


In [ ]:
filas_div = []
for (usuario, rango), grupo in generos.groupby(["usuario", "rango"], sort=False):
    conteo = grupo.drop_duplicates(["artista", "genero"])["genero"].value_counts()
    filas_div.append({
        "usuario": usuario, "rango": rango,
        "n_artistas_con_genero": grupo["artista"].nunique(),
        "n_generos": len(conteo),
        "entropia": round(sa.entropia_shannon(conteo), 4),
        "cuota_genero_top": round(float(conteo.iloc[0] / conteo.sum()), 4) if len(conteo) else 0.0,
    })
diversidad = pd.DataFrame(filas_div)
diversidad.to_csv(DIR_RES / "diversidad.csv", index=False)
display(diversidad)

In [ ]:
usuarios = sorted(df["usuario"].unique())
perfiles = {u: sa.perfil_generos(generos, usuario=u) for u in usuarios}
pd.concat(perfiles, names=["usuario", "genero"]).rename("n_artistas").to_csv(DIR_RES / "perfil_generos.csv")
limite = max(float(p.max()) for p in perfiles.values()) * 1.18
fig, axes = plt.subplots(1, len(usuarios), figsize=(10, 6), sharex=True)
for ax, usuario in zip(np.atleast_1d(axes), usuarios):
    perfil = perfiles[usuario].head(15).iloc[::-1]
    barras = ax.barh(perfil.index, perfil.values, height=0.65, color=PALETA[0])
    ax.bar_label(barras, padding=3, fontsize=10)
    ax.set_title(f"Top 15 etiquetas · {usuario.capitalize()}")
    ax.set_xlabel("Artistas distintos asociados a la etiqueta")
    ax.set_xlim(0, limite)
    ax.grid(axis="y", visible=False)
fig.tight_layout()
sa.guardar_figura(fig, "01_top_generos")
plt.show()

In [ ]:
tabla_generos = generos.drop_duplicates(["usuario", "rango", "artista", "genero"])
conteos = tabla_generos.groupby(["usuario", "rango", "genero"]).size().rename("n_asociaciones")
conteos.to_csv(DIR_RES / "generos_por_rango.csv")
top_generos = tabla_generos["genero"].value_counts().head(7).index.tolist()
fig, axes = plt.subplots(1, len(usuarios), figsize=(10, 5), sharey=True)
for ax, usuario in zip(np.atleast_1d(axes), usuarios):
    conteo = conteos.loc[usuario].unstack(fill_value=0)
    datos = conteo.reindex(columns=top_generos, fill_value=0).copy()
    datos["otras etiquetas"] = conteo.drop(columns=top_generos, errors="ignore").sum(axis=1)
    shares = datos.div(datos.sum(axis=1), axis=0).reindex(sa.ORDEN_RANGOS)
    shares.index = shares.index.map(CORTOS)
    shares.plot(kind="bar", stacked=True, ax=ax, color=PALETA, width=0.6,
                edgecolor=SUPERFICIE, linewidth=1.1, rot=0, legend=False)
    ax.set_ylabel("Proporción de asociaciones artista–etiqueta")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.set_title(usuario.capitalize())
    ax.grid(axis="x", visible=False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=10)
fig.suptitle("Mezcla de etiquetas por rango (no tiempo de escucha)")
fig.tight_layout(rect=(0, 0.16, 1, 0.94))
sa.guardar_figura(fig, "02_generos_por_rango")
plt.show()

## 3. Estabilidad temporal

In [ ]:
estabilidad = pd.concat(
    [sa.estabilidad_por_rangos(df, generos, u) for u in usuarios], ignore_index=True,
)
estabilidad.to_csv(DIR_RES / "estabilidad.csv", index=False)
display(estabilidad)
presencia = (
    df.groupby(["usuario", "artista"])["rango"].nunique().rename("n_rangos")
    .reset_index().groupby(["usuario", "n_rangos"]).size().unstack(fill_value=0)
)
presencia.to_csv(DIR_RES / "presencia_rangos.csv")
display(presencia)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
rangos = list(sa.ORDEN_RANGOS)
for ax, usuario in zip(axes, usuarios):
    matriz = pd.DataFrame(np.eye(len(rangos)), index=rangos, columns=rangos)
    for fila in estabilidad[estabilidad["usuario"] == usuario].itertuples(index=False):
        matriz.loc[fila.rango_a, fila.rango_b] = fila.jaccard_artistas
        matriz.loc[fila.rango_b, fila.rango_a] = fila.jaccard_artistas
    sns.heatmap(matriz, annot=True, fmt=".2f", vmin=0, vmax=1, cmap=SECUENCIAL,
                ax=ax, xticklabels=[CORTOS[r] for r in rangos],
                yticklabels=[CORTOS[r] for r in rangos], cbar=False)
    ax.set_title(f"Jaccard de artistas entre rangos · {usuario.capitalize()}")
    ax.set_xlabel("")
    ax.set_ylabel("")
fig.tight_layout()
sa.guardar_figura(fig, "03_estabilidad_rangos")
plt.show()

## 4. Enriquecimiento y valores nulos

In [ ]:
artistas_unicos = sorted(df["artista"].unique())
artistas, similitudes = sa.enriquecer_artistas(artistas_unicos)
artistas = artistas.merge(
    principales.rename("genero_principal"),
    left_on="artista",
    right_index=True,
    how="left",
)
generos_por_artista = df.drop_duplicates("artista").set_index("artista")["generos"]
artistas["generos"] = artistas["artista"].map(generos_por_artista)
nulos = sa.resumen_nulos(artistas)
nulos.to_csv(DIR_RES / "nulos.csv", index=False)
display(nulos)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
etiquetas_nulos = nulos["columna"].replace({"n_similares": "Sin vecinos similares"})
barras = ax.barh(etiquetas_nulos, nulos["pct_nulos"], color=PALETA[0], height=0.6)
ax.bar_label(barras, labels=[f"{v:.1f}%" for v in nulos["pct_nulos"]], padding=3)
ax.set_xlabel("Artistas sin dato o sin vecinos (%)")
ax.set_xlim(0, 112)
ax.set_xticks(range(0, 101, 20))
ax.set_title("Ausencia de información en los atributos enriquecidos")
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
fig.tight_layout()
sa.guardar_figura(fig, "09_nulos")
plt.show()

In [ ]:
artistas, inf_listeners = sa.imputar_popularidad(artistas, "listeners")
artistas, inf_playcount = sa.imputar_popularidad(artistas, "playcount")
artistas, inf_deezer = sa.imputar_popularidad(artistas, "deezer_fans")
informe_imputacion = pd.concat([inf_listeners, inf_playcount, inf_deezer], ignore_index=True)
display(informe_imputacion)

pares = artistas.dropna(subset=["listeners", "deezer_fans"])
rho_deezer, p_deezer = spearmanr(pares["listeners"], pares["deezer_fans"])
print(f"Spearman listeners (Last.fm) vs fans (Deezer): rho={rho_deezer:.3f}, p={p_deezer:.3g}, n={len(pares)}")


## 5. Compatibilidad entre usuarios

In [ ]:
compat = sa.tabla_compatibilidad(df, generos)
compat.to_csv(DIR_RES / "compatibilidad.csv", index=False)
display(compat)
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(compat))
ancho = 0.32
b1 = ax.bar(x - ancho / 2, compat["jaccard_artistas"], ancho, label="Jaccard (artistas)", color=PALETA[0])
b2 = ax.bar(x + ancho / 2, compat["coseno_generos"], ancho, label="Coseno (etiquetas)", color=PALETA[1], hatch="//")
for barras in (b1, b2):
    ax.bar_label(barras, fmt="%.3f", padding=3, fontsize=8)
ax.set_xticks(x, compat["rango"].map(CORTOS))
ax.set_ylim(0, 1)
ax.set_ylabel("Similitud")
ax.set_title("Compatibilidad musical entre los dos usuarios")
ax.grid(axis="x", visible=False)
ax.legend()
fig.tight_layout()
sa.guardar_figura(fig, "04_compatibilidad")
plt.show()

## 6. Popularidad frente a la posición en el ranking

In [ ]:
datos_pop = sa.unir_enriquecimiento(df, artistas)
tabla_spearman = sa.spearman_por_grupo(datos_pop)
tabla_spearman.to_csv(DIR_RES / "popularidad_spearman.csv", index=False)
display(tabla_spearman)
fig, ax = plt.subplots(figsize=(8, 5))
for usuario, marcador in zip(usuarios, ["o", "^"]):
    sub = datos_pop[(datos_pop["usuario"] == usuario) & (datos_pop["rango"] == "varios_anios")]
    sub = sub[sub["listeners"].notna() & (sub["listeners"] > 0)]
    ax.scatter(sub["posicion"], sub["listeners"], s=55, alpha=0.8, marker=marcador,
               label=usuario.capitalize(), color=COLORES_USUARIO[usuario],
               edgecolor=SUPERFICIE, linewidth=0.8)
ax.set_yscale("log")
ax.set_xlabel("Posición en el ranking (1 = mayor afinidad)")
ax.set_ylabel("Oyentes en Last.fm (escala logarítmica)")
ax.set_title("Popularidad frente al ranking de largo plazo (casos observados)")
ax.legend()
fig.tight_layout()
sa.guardar_figura(fig, "05_popularidad_rank")
plt.show()

## 7. Afinidad usuario–artista

In [ ]:
candidatos = sa.generar_candidatos(df, similitudes)
similitudes_candidatos = sa.enriquecer_similitudes(candidatos)
similitudes_completas = pd.concat([similitudes, similitudes_candidatos], ignore_index=True)
afinidad = sa.calcular_afinidad(df, similitudes_completas, candidatos)
afinidad["media"] = afinidad[usuarios].mean(axis=1, skipna=False)
afinidad = afinidad.sort_values(["media", "artista_candidato"], ascending=[False, True]).reset_index(drop=True)
afinidad.to_csv(DIR_RES / "afinidad_resumen.csv", index=False)
cobertura_afinidad = {"n_candidatos": len(candidatos), "n_puntuables": int(afinidad["media"].notna().sum())}
print(cobertura_afinidad)
display(afinidad.drop(columns="artista_candidato"))
loo = sa.evaluar_afinidad_loo(df, similitudes_completas, candidatos)
display(loo)

In [ ]:
top = afinidad.dropna(subset=["media"]).head(10).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(top))
alto = 0.32
for desplazamiento, usuario, trama in zip((-alto / 2, alto / 2), usuarios, ["", "//"]):
    barras = ax.barh(y + desplazamiento, top[usuario], alto, label=usuario.capitalize(),
                     color=COLORES_USUARIO[usuario], hatch=trama)
    ax.bar_label(barras, fmt="%.3f", padding=3, fontsize=8)
ax.set_yticks(y, [f"C{i:02d}" for i in range(1, len(top) + 1)])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("Afinidad estimada (no probabilidad)")
ax.set_title(f"Candidatos nuevos: {cobertura_afinidad['n_puntuables']}/{len(candidatos)} puntuables")
if top.empty:
    ax.text(0.5, 0.5, "Sin candidatos con información suficiente", ha="center", transform=ax.transAxes)
ax.legend()
ax.grid(axis="y", visible=False)
fig.tight_layout()
sa.guardar_figura(fig, "06_afinidad")
plt.show()

## 8. Agrupamiento de artistas por género

In [ ]:
X_df, info_clusters = sa.matriz_dummy_artistas(generos)
info_clusters["artistas_sin_genero"] = int(df["artista"].nunique() - generos["artista"].nunique())
print(info_clusters)
tabla_k, k_elegido = sa.elegir_k(X_df.values)
tabla_k.to_csv(DIR_RES / "seleccion_k.csv", index=False)
display(tabla_k)
print("k elegido:", k_elegido)
fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
for ax, columna, etiqueta in zip(axes, ["inercia", "silhouette"], ["Inercia", "Coeficiente de silueta"]):
    ax.plot(tabla_k["k"], tabla_k[columna], marker="o", markersize=6, linewidth=2, color=PALETA[0])
    ax.axvline(k_elegido, linestyle="--", color="#52514e", label=f"k elegido = {k_elegido}")
    ax.set_ylabel(etiqueta)
    ax.legend(fontsize=8)
axes[-1].set_xlabel("Número de grupos (k)")
axes[-1].set_xticks(tabla_k["k"])
axes[0].set_title("Selección de k: dos criterios en escalas separadas")
fig.tight_layout()
sa.guardar_figura(fig, "08_seleccion_k")
plt.show()

In [ ]:
etiquetas, modelo = sa.ajustar_clusters(X_df.values, k_elegido)
perfil_clusters = sa.etiquetar_clusters(X_df, etiquetas)
perfil_clusters.to_csv(DIR_RES / "clusters_resumen.csv", index=False)
display(perfil_clusters)
coords, varianza = sa.proyectar_pca(X_df.values)
filas_panel = int(np.ceil(k_elegido / 2))
fig, axes = plt.subplots(filas_panel, 2, figsize=(10, 3 * filas_panel), sharex=True, sharey=True, squeeze=False)
for cluster, ax in enumerate(axes.flat):
    if cluster >= k_elegido:
        ax.set_visible(False)
        continue
    mascara = etiquetas == cluster
    ax.scatter(coords[~mascara, 0], coords[~mascara, 1], s=28, c="#c3c2b7", alpha=0.5, label="Otros grupos")
    ax.scatter(coords[mascara, 0], coords[mascara, 1], s=52, color=PALETA[0], edgecolor=SUPERFICIE, linewidth=0.6, label="Grupo resaltado")
    centro = coords[mascara].mean(axis=0)
    ax.scatter(*centro, marker="X", s=100, c="#0b0b0b", label="Centroide proyectado")
    ax.set_title(f"Grupo {cluster} · n = {int(mascara.sum())}")
    ax.set_xlabel("Componente 1")
    ax.set_ylabel("Componente 2")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8)
fig.suptitle(f"Grupos en PCA · varianza explicada: {varianza:.1%}")
fig.tight_layout(rect=(0, 0.05, 1, 0.95))
sa.guardar_figura(fig, "07_clustering_pca")
plt.show()

In [ ]:
generos_frecuentes = X_df.mean(axis=0).sort_values(ascending=False).head(12).index
prevalencia = pd.DataFrame({c: X_df.loc[etiquetas == c, generos_frecuentes].mean(axis=0) for c in range(k_elegido)}).T
prevalencia.to_csv(DIR_RES / "clusters_prevalencia.csv", index_label="cluster")
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(prevalencia, annot=True, fmt=".2f", cmap=SECUENCIAL, vmin=0, vmax=1, ax=ax,
            cbar_kws={"label": "Proporción de artistas"})
ax.set_xlabel("Etiqueta")
ax.set_ylabel("Grupo")
ax.set_title("Prevalencia de las etiquetas frecuentes por grupo")
fig.tight_layout()
sa.guardar_figura(fig, "07b_perfil_clusters")
plt.show()
artista_usuario = df.drop_duplicates(["artista", "usuario"])[["artista", "usuario"]]
clusters_df = pd.DataFrame({"artista": X_df.index, "cluster": etiquetas}).merge(artista_usuario, on="artista", how="left")
clusters_usuarios = pd.crosstab(clusters_df["cluster"], clusters_df["usuario"])
clusters_usuarios.to_csv(DIR_RES / "clusters_usuarios.csv")
display(clusters_usuarios)

## 9. Síntesis y exportación de métricas

In [ ]:
afinidad_anon = afinidad.dropna(subset=["media"]).head(10).copy()
afinidad_anon.insert(0, "candidato", [f"C{i:02d}" for i in range(1, len(afinidad_anon) + 1)])
afinidad_anon[["candidato", *usuarios, "media"]].to_csv(DIR_RES / "candidatos_resumen.csv", index=False)
metricas = {
    "meta": {
        "fecha_ejecucion": pd.Timestamp.now().strftime("%Y-%m-%d"),
        "usuarios": usuarios, "n_filas": len(df),
        "n_artistas_unicos": int(df["artista"].nunique()),
    },
    "validacion": validacion.to_dict(orient="records"),
    "resumen": resumen.to_dict(orient="records"),
    "diversidad": diversidad.to_dict(orient="records"),
    "estabilidad": estabilidad.to_dict(orient="records"),
    "presencia_rangos": presencia.reset_index().to_dict(orient="records"),
    "compatibilidad": compat.to_dict(orient="records"),
    "popularidad": tabla_spearman.to_dict(orient="records"),
    "afinidad": {
        "cobertura": cobertura_afinidad, "loo": loo.to_dict(orient="records"),
        "top10": afinidad_anon[["candidato", *usuarios, "media"]].to_dict(orient="records"),
    },
    "clustering": {
        "info": info_clusters, "k": int(k_elegido),
        "seleccion_k": tabla_k.to_dict(orient="records"),
        "perfiles": perfil_clusters.to_dict(orient="records"),
        "usuarios": clusters_usuarios.reset_index().to_dict(orient="records"),
        "varianza_pca": round(varianza, 4),
    },
    "nulos": nulos.to_dict(orient="records"),
    "imputacion": informe_imputacion.to_dict(orient="records"),
    "deezer_vs_lastfm": {"rho": float(rho_deezer), "p": float(p_deezer), "n": int(len(pares))},
}
sa.exportar_metricas(metricas, DIR_RES / "metricas.json")
import json
meta_extraccion = json.loads((DIR_RES / "extraccion_meta.json").read_text(encoding="utf-8"))
for usuario, grupo in df.groupby("usuario"):
    if not isinstance(meta_extraccion.get(usuario), dict):
        meta_extraccion[usuario] = {"fecha": None}
    meta_extraccion[usuario].update({
        "fecha_validacion": metricas["meta"]["fecha_ejecucion"],
        "filas": len(grupo), "artistas_unicos": int(grupo["artista"].nunique()),
        "rangos": list(sa.ORDEN_RANGOS),
        "filas_con_generos": int(grupo["generos"].str.strip().ne("").sum()),
    })
sa.exportar_metricas(meta_extraccion, DIR_RES / "extraccion_meta.json")
print("Métricas exportadas:", list(metricas))

## 10. Conclusiones e interpretación

- Los 298 registros reúnen 140 artistas distintos, con 5 compartidos: el Jaccard global es 0,0357 y el coseno de etiquetas 0,4686. Las métricas comparan aspectos distintos y no son probabilidades de compatibilidad.
- Los perfiles de etiquetas son más parecidos entre las ventanas de cada usuario que los conjuntos de artistas. Las ventanas se solapan; no constituyen observaciones longitudinales independientes.
- Jorge presenta mayor entropía de etiquetas en los tres rangos, pero las etiquetas colaborativas contienen sinónimos y procedencias. Sus proporciones no son tiempo de escucha ni una medida objetiva de diversidad musical.
- La cobertura de Last.fm/Deezer y la comprobación de nombres condicionan los contrastes. Las correlaciones usan valores observados, nunca imputados; los resultados no calculables se exportan como `null`.
- La afinidad es una heurística basada en vecinos, no una probabilidad ni una predicción supervisada. Un candidato no escuchado no equivale a un ejemplo negativo y la comparación leave-one-out no es validación independiente.
- El agrupamiento es exploratorio: la matriz conserva 133 artistas y 52 etiquetas; la regla elige dos grupos con silueta 0,1508. La proyección PCA explica solo el 26,42 % de la varianza y no valida la partición.
- El informe de `doc/main.pdf` desarrolla las cifras de `resultados/metricas.json`, la cobertura efectiva y las limitaciones. Las respuestas personales y la copia ejecutada del cuaderno permanecen en `cache/`, fuera del repositorio.